In [ ]:
# ==============================================================================
# 🏆 IEEE 118-Bus R-PINN Proposed Method - Mean-Loss Aligned Edition
# 特点：聚合方式由 Sum 改为 Mean，严格对齐 69 节点铂金基线
# 核心：ARS | 节点身份掩码 | 双状态评估 | 动态物理权重
# 修复：彻底剔除了物理算子中导致非传感器节点相角飞升(22+ rad)的多余 π/180 转换！
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandapower.networks as nw
import pandapower as pp
import matplotlib.pyplot as plt
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 R-PINN 118节点 Mean-Loss 版就绪 | 算力: {device}")

NUM_NODES = 118
# 15% 观测点 (包含平衡节点 68)
obs_indices = [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]
print(f"📡 锁定的 15% PMU 观测点: {obs_indices}")

# ------------------------------------------
# 2. 物理矩阵与节点身份掩码
# ------------------------------------------
net = nw.case118()
pp.runpp(net)
Ybus = net._ppc['internal']['Ybus']
G_tensor = torch.tensor(Ybus.real.toarray(), dtype=torch.float32).to(device)
B_tensor = torch.tensor(Ybus.imag.toarray(), dtype=torch.float32).to(device)

slack_buses = net.ext_grid.bus.values      # 索引 68
pv_buses = net.gen.bus.values
load_buses = net.load.bus.values
all_buses = np.arange(NUM_NODES)
zero_inj_buses = np.setdiff1d(all_buses, np.concatenate([slack_buses, pv_buses, load_buses]))

# P 掩码：除 Slack 外所有节点的有功 P 受约束
mask_p_np = np.ones(NUM_NODES)
mask_p_np[slack_buses] = 0.0

# Q 掩码：Slack 和 PV 节点的无功 Q 不受约束
mask_q_np = np.zeros(NUM_NODES)
mask_q_np[load_buses] = 1.0
mask_q_np[zero_inj_buses] = 1.0

mask_p = torch.tensor(mask_p_np, dtype=torch.float32).to(device)
mask_q = torch.tensor(mask_q_np, dtype=torch.float32).to(device)

# ------------------------------------------
# 3. 模型与损失 (Mean‑Loss)
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    # ✅ 致命错误修复：传入的 theta_pred 已经是弧度了，绝不能再乘 (np.pi / 180.0)！
    # 直接计算角度差
    dT = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=118):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        # 👑 保持你修改过的正确 ARS 配置：Bias 0.5236, Scale 0.05
        theta_pred = out[:, self.node_num:] * 0.05 + 0.5236     
    
        vm_pred = vm_pred.clone(); theta_pred = theta_pred.clone()
    
        # 🚨 物理参考系对齐：必须用真实数据锚定平衡节点
        vm_pred[:, 68] = 1.035      
        theta_pred[:, 68] = 0.5236  
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        # 潮流计算
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        # 物理损失：按有效节点取平均（屏蔽 Slack）
        diff_p = (P_calc - P_real) ** 2
        diff_q = (Q_calc - Q_real) ** 2
        bs = diff_p.shape[0]
        P_loss = torch.sum(diff_p * mask_p) / (torch.sum(mask_p) * bs)
        Q_loss = torch.sum(diff_q * mask_q) / (torch.sum(mask_q) * bs)
        
        # 观测损失：双状态 Mean
        obs_loss_v = self.mse(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        obs_loss = obs_loss_v + obs_loss_t
        
        # 惩罚项：Mean (平方 Relu)
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.10)
        penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
        
        # 为了防范梯度霸权，适度下调 obs_weight 是合理的，这里我先为你设为 5e4
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_idx_list, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_idx_list:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载
# ------------------------------------------
print("📂 正在进行数据装载与切片...")
data_3d = np.load(r'ieee118_dataset/ieee118_data_50k.npy')
m = len(data_3d)
P_inj = -data_3d[:, :, 0] / 100.0  
Q_inj = -data_3d[:, :, 1] / 100.0
V_val = data_3d[:, :, 2]
T_rad = np.deg2rad(data_3d[:, :, 3]) 

X_raw = np.concatenate([P_inj, Q_inj], axis=1)
Y_label = np.concatenate([V_val, T_rad], axis=1)

train_size = 40000
X_tr_raw, X_te_raw = X_raw[:train_size], X_raw[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_inj[:train_size], Q_inj[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

class GridDataset(Dataset):
    def __init__(self, x, y, p, q): self.x, self.y, self.p, self.q = x, y, p, q
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.p[i], self.q[i]

train_loader = DataLoader(GridDataset(
    torch.tensor(X_tr_norm, dtype=torch.float32), torch.tensor(Y_tr, dtype=torch.float32),
    torch.tensor(P_tr_phys, dtype=torch.float32), torch.tensor(Q_tr_phys, dtype=torch.float32)), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 训练 (动态权重 + Mean‑Loss)
# ------------------------------------------
model = PowerGridPINN(node_num=118).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 300 轮 Mean-Loss 训练 (已修复物理算子与梯度霸权)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        loss = loss_fn(vp, tp, bp, bq, by[:, :118], by[:, 118:], lambda_p, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态评估 (双状态 + RMSE)
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.tensor(X_te_norm, dtype=torch.float32).to(device)
    ty = torch.tensor(Y_te, dtype=torch.float32).to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all = ty[:, :118]
    t_true_all = ty[:, 118:]
    
    err_v = (v_pred_all - v_true_all).cpu().numpy()
    err_t = (t_pred_all - t_true_all).cpu().numpy()
    mae_v = np.mean(np.abs(err_v))
    rmse_v = np.sqrt(np.mean(err_v**2))
    mae_t = np.mean(np.abs(err_t))
    rmse_t = np.sqrt(np.mean(err_t**2))

print("\n" + "="*65)
print(f"🏆 [118-Bus Mean-Loss] 基态对账单")
print(f"🌍 Voltage (V) MAE  : {mae_v:.6e} p.u. | RMSE: {rmse_v:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t:.6e} rad  | RMSE: {rmse_t:.6e} rad")
print("="*65)

# ------------------------------------------
# 7. 双子图展示
# ------------------------------------------
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-paper')

sid = 10
v_t_plot = v_true_all[sid].cpu().numpy()
v_p_plot = v_pred_all[sid].cpu().numpy()
t_t_plot = t_true_all[sid].cpu().numpy()
t_p_plot = t_pred_all[sid].cpu().numpy()
nodes = np.arange(118)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), dpi=600, sharex=True)
ax1.axvspan(0, 117, color='gray', alpha=0.05, label='Blind Zone (85%)')
ax1.plot(nodes, v_t_plot, 'k-', label='Ground Truth', linewidth=2.0)
ax1.plot(nodes, v_p_plot, 'r--', label='Proposed R-PINN', linewidth=1.5)
ax1.scatter(obs_indices, v_t_plot[obs_indices], color='blue', marker='*', s=150, label='PMU Sensors')
ax1.set_title(f"IEEE 118-Bus Profile: (a) Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', fontsize=11)

ax2.axvspan(0, 117, color='gray', alpha=0.05)
ax2.plot(nodes, t_t_plot, 'k-', linewidth=2.0)
ax2.plot(nodes, t_p_plot, 'b--', label='Proposed R-PINN (Theta)', linewidth=1.5)
ax2.scatter(obs_indices, t_t_plot[obs_indices], color='blue', marker='*', s=150)
ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} rad | RMSE: {rmse_t:.4e} rad)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Bus Index", fontsize=12)
ax2.set_ylabel("Phase Angle (rad)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('ieee118_platinum_reconstruction.png', bbox_inches='tight', dpi=600)
plt.show()

print("✅ 118 节点 Mean-Loss 版训练完毕，可直接用于论文主实验。")

🚀 R-PINN 118节点 Mean-Loss 版就绪 | 算力: cuda
📡 锁定的 15% PMU 观测点: [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]
📂 正在进行数据装载与切片...
🔥 开始 300 轮 Mean-Loss 训练 (已修复物理算子与梯度霸权)...
Epoch 0   | Avg Loss: 1.3204e+06 | LR: 9.999729e-04


In [ ]:
# ==============================================================================
# 📋 IEEE 118-Bus 终极对账单 (电压 + 相角)
# 使用已训练好且仍在内存中的 v_true_all, v_pred_all, t_true_all, t_pred_all
# ==============================================================================

import numpy as np
import torch

# 选择要查看的样本编号
sid = 10

# 从 GPU 拷贝到 CPU 并转为 numpy（如果还在 GPU 上）
v_true = v_true_all[sid].cpu().numpy()
v_pred = v_pred_all[sid].cpu().numpy()
t_true = t_true_all[sid].cpu().numpy()
t_pred = t_pred_all[sid].cpu().numpy()

# 打印表头
print("\n" + "=" * 110)
print(f"📝 118-Bus Full Node Report (Sample #{sid})")
print("=" * 110)
print(f"{'Bus':<5} | {'V_True':<10} | {'V_Pred':<10} | {'V_Bias':<10} | {'θ_True (rad)':<14} | {'θ_Pred (rad)':<14} | {'θ_Bias (rad)'}")
print("-" * 110)

# 依次打印每个节点的信息
for i in range(118):
    tag = "★" if i in obs_indices else "  "   # 标记传感器节点
    v_bias = v_pred[i] - v_true[i]
    t_bias = t_pred[i] - t_true[i]
    print(f"{i:<3} {tag} | {v_true[i]:<10.4f} | {v_pred[i]:<10.4f} | {v_bias:+.2e}   | {t_true[i]:<14.4f} | {t_pred[i]:<14.4f} | {t_bias:+.2e}")

print("-" * 110)

In [ ]:
# ==============================================================================
# 🏆 IEEE 118-Bus Ablation Case 1: Pure Data-Driven (Baseline)
# 变量控制：【保留】300轮余弦退火、ARS(+0.5236)、硬锚定、Mean-Loss
# 变量控制：【修改】物理损失权重 lambda_p 恒定为 0.0 (彻底剥离物理约束)
# 目的：验证纯黑盒模型在基础状态下的拟合能力。
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandapower.networks as nw
import pandapower as pp
import matplotlib.pyplot as plt
import random
import os
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定 (严格对齐)
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 118节点消融实验 (Case 1: 纯数据驱动 对齐版) 启动 | 算力: {device}")

NUM_NODES = 118
obs_indices = [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]

# ------------------------------------------
# 2. 物理矩阵与节点身份掩码 (严格对齐)
# ------------------------------------------
net = nw.case118()
pp.runpp(net)
Ybus = net._ppc['internal']['Ybus']
G_tensor = torch.tensor(Ybus.real.toarray(), dtype=torch.float32).to(device)
B_tensor = torch.tensor(Ybus.imag.toarray(), dtype=torch.float32).to(device)

slack_buses = net.ext_grid.bus.values      # 索引 68
pv_buses = net.gen.bus.values
load_buses = net.load.bus.values
all_buses = np.arange(NUM_NODES)
zero_inj_buses = np.setdiff1d(all_buses, np.concatenate([slack_buses, pv_buses, load_buses]))

mask_p_np = np.ones(NUM_NODES)
mask_p_np[slack_buses] = 0.0

mask_q_np = np.zeros(NUM_NODES)
mask_q_np[load_buses] = 1.0
mask_q_np[zero_inj_buses] = 1.0

mask_p = torch.tensor(mask_p_np, dtype=torch.float32).to(device)
mask_q = torch.tensor(mask_q_np, dtype=torch.float32).to(device)

# ------------------------------------------
# 3. 模型与损失封装 (保留 ARS 与 Mean-Loss 结构)
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    dT = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=118):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        # 保留 ARS 先验：118节点相角基准必须是 0.5236
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        theta_pred = out[:, self.node_num:] * 0.05 + 0.5236     
        
        vm_pred = vm_pred.clone(); theta_pred = theta_pred.clone()
        vm_pred[:, 68] = 1.035; theta_pred[:, 68] = 0.5236      # 平衡节点硬锚定必须在
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        diff_p = (P_calc - P_real) ** 2
        diff_q = (Q_calc - Q_real) ** 2
        bs = diff_p.shape[0]
        P_loss = torch.sum(diff_p * mask_p) / (torch.sum(mask_p) * bs)
        Q_loss = torch.sum(diff_q * mask_q) / (torch.sum(mask_q) * bs)
        
        obs_loss_v = self.mse(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        obs_loss = obs_loss_v + obs_loss_t
        
        # 上限 1.10 的越限惩罚
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.10)
        penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
        
        # 注意：这里的 p_weight 在外部传进来时将被锁定为 0.0
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_idx_list, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_idx_list:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载 (严格对齐 40k)
# ------------------------------------------
print("📂 加载 118 节点数据...")
data_3d = np.load(r'ieee118_dataset/ieee118_data_50k.npy')
P_inj = -data_3d[:, :, 0] / 100.0  
Q_inj = -data_3d[:, :, 1] / 100.0
V_val = data_3d[:, :, 2]
T_rad = np.deg2rad(data_3d[:, :, 3]) 

X_raw = np.concatenate([P_inj, Q_inj], axis=1)
Y_label = np.concatenate([V_val, T_rad], axis=1)

train_size = 40000
X_tr_raw, X_te_raw = X_raw[:train_size], X_raw[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_inj[:train_size], Q_inj[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

class GridDataset(Dataset):
    def __init__(self, x, y, p, q): self.x, self.y, self.p, self.q = x, y, p, q
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.p[i], self.q[i]

train_loader = DataLoader(GridDataset(
    torch.tensor(X_tr_norm, dtype=torch.float32), torch.tensor(Y_tr, dtype=torch.float32),
    torch.tensor(P_tr_phys, dtype=torch.float32), torch.tensor(Q_tr_phys, dtype=torch.float32)), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 训练 (唯一变量点：剥离物理约束)
# ------------------------------------------
model = PowerGridPINN(node_num=118).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 300 轮 Case 1 (纯数据驱动) 绝对对齐训练...")
for epoch in range(300):
    # ❌ 核心控制变量：彻底封印物理规则，仅靠数据拟合
    lambda_p = 0.0  
    
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        # 👑 严格对齐主实验的 obs_weight=5e6
        loss = loss_fn(vp, tp, bp, bq, by[:, :118], by[:, 118:], lambda_p, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态测试集评估
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.tensor(X_te_norm, dtype=torch.float32).to(device)
    ty = torch.tensor(Y_te, dtype=torch.float32).to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
    v_pred_base, t_pred_base = model(t_mask)
    v_true_base = ty[:, :118]
    t_true_base = ty[:, 118:]
    
    err_v = (v_pred_base - v_true_base).cpu().numpy()
    err_t = (t_pred_base - t_true_base).cpu().numpy()
    v_mae = np.mean(np.abs(err_v))
    v_rmse = np.sqrt(np.mean(err_v**2))
    t_mae = np.mean(np.abs(err_t))
    t_rmse = np.sqrt(np.mean(err_t**2))

print("\n" + "="*60)
print(f"🏆 基态对账单 (Case 1: Pure Data-Driven)")
print(f"   Voltage MAE : {v_mae:.6e}  | RMSE : {v_rmse:.6e}")
print(f"   Phase Angle MAE : {t_mae:.6e}  | RMSE : {t_rmse:.6e}")
print("="*60)

In [ ]:
# ==============================================================================
# 🏆 IEEE 118-Bus Ablation Study - Case: Without ARS (No ARS)
# 变量控制：【保留】Mean-Loss, 节点身份掩码, 双状态评估, 动态物理权重, 硬锚定 (1.035, 0.5236), obs_weight=5e4
# 变量控制：【剔除】ARS 残差缩放层 —— 电压和相角直接从网络原始输出
# 修复：已彻底剔除物理算子中导致相角飞升的多余 π/180 转换。
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandapower.networks as nw
import pandapower as pp
import matplotlib.pyplot as plt
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 [No ARS] 118节点消融实验 (修复对齐版) | 算力: {device}")

NUM_NODES = 118
obs_indices = [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]

# ------------------------------------------
# 2. 物理矩阵与节点身份掩码
# ------------------------------------------
net = nw.case118()
pp.runpp(net)
Ybus = net._ppc['internal']['Ybus']
G_tensor = torch.tensor(Ybus.real.toarray(), dtype=torch.float32).to(device)
B_tensor = torch.tensor(Ybus.imag.toarray(), dtype=torch.float32).to(device)

slack_buses = net.ext_grid.bus.values      
pv_buses = net.gen.bus.values
load_buses = net.load.bus.values
all_buses = np.arange(NUM_NODES)
zero_inj_buses = np.setdiff1d(all_buses, np.concatenate([slack_buses, pv_buses, load_buses]))

mask_p_np = np.ones(NUM_NODES)
mask_p_np[slack_buses] = 0.0
mask_q_np = np.zeros(NUM_NODES)
mask_q_np[load_buses] = 1.0
mask_q_np[zero_inj_buses] = 1.0

mask_p = torch.tensor(mask_p_np, dtype=torch.float32).to(device)
mask_q = torch.tensor(mask_q_np, dtype=torch.float32).to(device)

# ------------------------------------------
# 3. 模型：彻底移除 ARS 残差缩放
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    # ✅ 修复：直接计算角度差，绝不能再乘 pi/180！
    dT = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

class PowerGridPINN_NoARS(nn.Module):
    def __init__(self, node_num=118):
        super(PowerGridPINN_NoARS, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2)
        )
    def forward(self, x):
        out = self.net(x)
        # ❌ 消融核心：彻底移除 ARS 残差缩放
        vm_pred = torch.sigmoid(out[:, :self.node_num]) * 1.2
        theta_pred = out[:, self.node_num:]
        
        vm_pred = vm_pred.clone()
        theta_pred = theta_pred.clone()
        # 🛡️ 硬锚定 Slack Bus：必须对齐数据集的真实基准
        vm_pred[:, 68] = 1.035
        theta_pred[:, 68] = 0.5236
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight):
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        diff_p = (P_calc - P_real) ** 2
        diff_q = (Q_calc - Q_real) ** 2
        bs = diff_p.shape[0]
        P_loss = torch.sum(diff_p * mask_p) / (torch.sum(mask_p) * bs)
        Q_loss = torch.sum(diff_q * mask_q) / (torch.sum(mask_q) * bs)
        
        obs_loss_v = self.mse(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        obs_loss = obs_loss_v + obs_loss_t
        
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.10)
        penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
        
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_idx_list, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_idx_list:
        mask_bx[:, idx] = batch_x[:, idx]
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]
    return mask_bx

# ------------------------------------------
# 4. 数据装载
# ------------------------------------------
data_3d = np.load(r'ieee118_dataset/ieee118_data_50k.npy')
P_inj = -data_3d[:, :, 0] / 100.0
Q_inj = -data_3d[:, :, 1] / 100.0
V_val = data_3d[:, :, 2]
T_rad = np.deg2rad(data_3d[:, :, 3])

X_raw = np.concatenate([P_inj, Q_inj], axis=1)
Y_label = np.concatenate([V_val, T_rad], axis=1)

train_size = 40000
X_tr_raw, X_te_raw = X_raw[:train_size], X_raw[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_inj[:train_size], Q_inj[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

class GridDataset(Dataset):
    def __init__(self, x, y, p, q): self.x, self.y, self.p, self.q = x, y, p, q
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.p[i], self.q[i]

train_loader = DataLoader(GridDataset(
    torch.tensor(X_tr_norm, dtype=torch.float32), torch.tensor(Y_tr, dtype=torch.float32),
    torch.tensor(P_tr_phys, dtype=torch.float32), torch.tensor(Q_tr_phys, dtype=torch.float32)),
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 训练
# ------------------------------------------
model = PowerGridPINN_NoARS(node_num=118).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 300 轮训练 (No ARS)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        # ✅ 对齐主实验：降低 obs_weight 避免梯度霸权
        loss = loss_fn(vp, tp, bp, bq, by[:, :118], by[:, 118:], lambda_p, obs_weight=5e6)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 评估
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.tensor(X_te_norm, dtype=torch.float32).to(device)
    ty = torch.tensor(Y_te, dtype=torch.float32).to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all = ty[:, :118]
    t_true_all = ty[:, 118:]
    
    err_v = (v_pred_all - v_true_all).cpu().numpy()
    err_t = (t_pred_all - t_true_all).cpu().numpy()
    mae_v = np.mean(np.abs(err_v))
    rmse_v = np.sqrt(np.mean(err_v**2))
    mae_t = np.mean(np.abs(err_t))
    rmse_t = np.sqrt(np.mean(err_t**2))

print("\n" + "="*65)
print(f"🏆 [No ARS] 测试集评估结果")
print(f"🌍 Voltage (V) MAE  : {mae_v:.6e} p.u. | RMSE: {rmse_v:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t:.6e} rad  | RMSE: {rmse_t:.6e} rad")
print("="*65)

# ------------------------------------------
# 7. 画图
# ------------------------------------------
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-paper')

sid = 10
v_t_plot = v_true_all[sid].cpu().numpy()
v_p_plot = v_pred_all[sid].cpu().numpy()
t_t_plot = t_true_all[sid].cpu().numpy()
t_p_plot = t_pred_all[sid].cpu().numpy()
nodes = np.arange(118)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), dpi=600, sharex=True)
ax1.axvspan(0, 117, color='gray', alpha=0.05, label='Blind Zone (85%)')
ax1.plot(nodes, v_t_plot, 'k-', label='Ground Truth', linewidth=2.0)
ax1.plot(nodes, v_p_plot, 'r--', label='No ARS', linewidth=1.5)
ax1.scatter(obs_indices, v_t_plot[obs_indices], color='blue', marker='*', s=150, label='PMU Sensors')
ax1.set_title(f"No ARS - Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', fontsize=11)

ax2.axvspan(0, 117, color='gray', alpha=0.05)
ax2.plot(nodes, t_t_plot, 'k-', linewidth=2.0)
ax2.plot(nodes, t_p_plot, 'b--', label='No ARS Theta', linewidth=1.5)
ax2.scatter(obs_indices, t_t_plot[obs_indices], color='blue', marker='*', s=150)
ax2.set_title(f"No ARS - Phase Angle\n(MAE: {mae_t:.4e} rad | RMSE: {rmse_t:.4e} rad)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Bus Index", fontsize=12)
ax2.set_ylabel("Phase Angle (rad)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('ieee118_ablation_no_ars.png', bbox_inches='tight', dpi=600)
plt.show()

print("✅ 无 ARS 消融实验已完成！")

In [ ]:
# ==============================================================================
# 🏆 IEEE 118-Bus Ablation Study - Case 3: PINN without PSC (Sign Mismatch)
# 变量控制：【保留】主模型所有的网络结构、Mean-Loss、动态权重、硬锚定、优化策略
# 变量控制：【剔除】物理语义对齐 (PSC)。故意不加负号，将负荷消耗直接作为正向注入。
# 目的：证明数据语义与物理法则未严格对齐会导致严重的梯度冲突与重构失败。
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandapower.networks as nw
import pandapower as pp
import matplotlib.pyplot as plt
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定 (严格对齐主模型)
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 [Case 3: w/o PSC] 118节点消融实验就绪 | 算力: {device}")

NUM_NODES = 118
obs_indices = [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]

# ------------------------------------------
# 2. 物理矩阵与节点身份掩码 (严格对齐主模型)
# ------------------------------------------
net = nw.case118()
pp.runpp(net)
Ybus = net._ppc['internal']['Ybus']
G_tensor = torch.tensor(Ybus.real.toarray(), dtype=torch.float32).to(device)
B_tensor = torch.tensor(Ybus.imag.toarray(), dtype=torch.float32).to(device)

slack_buses = net.ext_grid.bus.values      
pv_buses = net.gen.bus.values
load_buses = net.load.bus.values
all_buses = np.arange(NUM_NODES)
zero_inj_buses = np.setdiff1d(all_buses, np.concatenate([slack_buses, pv_buses, load_buses]))

mask_p_np = np.ones(NUM_NODES)
mask_p_np[slack_buses] = 0.0

mask_q_np = np.zeros(NUM_NODES)
mask_q_np[load_buses] = 1.0
mask_q_np[zero_inj_buses] = 1.0

mask_p = torch.tensor(mask_p_np, dtype=torch.float32).to(device)
mask_q = torch.tensor(mask_q_np, dtype=torch.float32).to(device)

# ------------------------------------------
# 3. 模型与损失 (严格对齐主模型)
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    # ✅ 修复：直接计算角度差，绝不能再乘 pi/180！
    dT = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=118):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        theta_pred = out[:, self.node_num:] * 0.05 + 0.5236    
        vm_pred = vm_pred.clone(); theta_pred = theta_pred.clone()
        # 【关键保留】：Slack Bus 硬锚定必须留着，否则无法判断是基准漂移还是物理崩溃
        vm_pred[:, 68] = 1.035; theta_pred[:, 68] = 0.5236      
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        diff_p = (P_calc - P_real) ** 2
        diff_q = (Q_calc - Q_real) ** 2
        bs = diff_p.shape[0]
        P_loss = torch.sum(diff_p * mask_p) / (torch.sum(mask_p) * bs)
        Q_loss = torch.sum(diff_q * mask_q) / (torch.sum(mask_q) * bs)
        
        obs_loss_v = self.mse(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        obs_loss = obs_loss_v + obs_loss_t
        
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.10)
        penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
        
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_idx_list, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_idx_list:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载 (核心消融点：剥离 PSC)
# ------------------------------------------
print("📂 Loading Data & INTENTIONALLY IGNORING PSC Alignment...")
data_3d = np.load(r'ieee118_dataset/ieee118_data_50k.npy')
m = len(data_3d)

# ❌ 【唯一修改点】：致命错误再现！故意不加负号！
# 让负荷消耗（本来该是负数）被当成发电机正向注入喂给物理方程！
P_inj = data_3d[:, :, 0] / 100.0  
Q_inj = data_3d[:, :, 1] / 100.0  
V_val = data_3d[:, :, 2]
T_rad = np.deg2rad(data_3d[:, :, 3]) 

X_raw = np.concatenate([P_inj, Q_inj], axis=1)
Y_label = np.concatenate([V_val, T_rad], axis=1)

train_size = 40000
X_tr_raw, X_te_raw = X_raw[:train_size], X_raw[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_inj[:train_size], Q_inj[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

class GridDataset(Dataset):
    def __init__(self, x, y, p, q): self.x, self.y, self.p, self.q = x, y, p, q
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.p[i], self.q[i]

train_loader = DataLoader(GridDataset(
    torch.tensor(X_tr_norm, dtype=torch.float32), torch.tensor(Y_tr, dtype=torch.float32),
    torch.tensor(P_tr_phys, dtype=torch.float32), torch.tensor(Q_tr_phys, dtype=torch.float32)), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 训练 (严格保持主模型的 300 轮与动态权重)
# ------------------------------------------
model = PowerGridPINN(node_num=118).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 300 轮训练 (Case 3: 无 PSC 语义对齐)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000  # 保持主模型的动态权重跃迁
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        # 此时传进去的 bp, bq (即 P_real, Q_real) 符号是错的，将与物理公式强行打架
        loss = loss_fn(vp, tp, bp, bq, by[:, :118], by[:, 118:], lambda_p, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态评估
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.tensor(X_te_norm, dtype=torch.float32).to(device)
    ty = torch.tensor(Y_te, dtype=torch.float32).to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all = ty[:, :118]
    t_true_all = ty[:, 118:]
    
    err_v = (v_pred_all - v_true_all).cpu().numpy()
    err_t = (t_pred_all - t_true_all).cpu().numpy()
    mae_v = np.mean(np.abs(err_v))
    rmse_v = np.sqrt(np.mean(err_v**2))
    mae_t = np.mean(np.abs(err_t))
    rmse_t = np.sqrt(np.mean(err_t**2))

print("\n" + "="*65)
print(f"🏆 [Case 3: w/o PSC] 基态对账单")
print(f"🌍 Voltage (V) MAE  : {mae_v:.6e} p.u. | RMSE: {rmse_v:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t:.6e} rad  | RMSE: {rmse_t:.6e} rad")
print("="*65)

# ------------------------------------------
# 7. 学术绘图
# ------------------------------------------
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-paper')

sid = 10
v_t_plot = v_true_all[sid].cpu().numpy()
v_p_plot = v_pred_all[sid].cpu().numpy()
t_t_plot = t_true_all[sid].cpu().numpy()
t_p_plot = t_pred_all[sid].cpu().numpy()
nodes = np.arange(118)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), dpi=600, sharex=True)
ax1.axvspan(0, 117, color='gray', alpha=0.05, label='Blind Zone (85%)')
ax1.plot(nodes, v_t_plot, 'k-', label='Ground Truth', linewidth=2.0)
ax1.plot(nodes, v_p_plot, 'm-.', label='Case 3 (No PSC)', linewidth=1.5, alpha=0.9)
ax1.scatter(obs_indices, v_t_plot[obs_indices], color='blue', marker='*', s=150, label='PMU Sensors')
ax1.set_title(f"Ablation Case 3: (a) Voltage Magnitude with Semantic Mismatch\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', fontsize=11)

ax2.axvspan(0, 117, color='gray', alpha=0.05)
ax2.plot(nodes, t_t_plot, 'k-', linewidth=2.0)
ax2.plot(nodes, t_p_plot, 'm-.', label='Case 3 (No PSC) Theta', linewidth=1.5, alpha=0.9)
ax2.scatter(obs_indices, t_t_plot[obs_indices], color='blue', marker='*', s=150)
ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} rad | RMSE: {rmse_t:.4e} rad)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Bus Index", fontsize=12)
ax2.set_ylabel("Phase Angle (rad)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('ieee118_ablation_case3_no_psc.png', bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ==============================================================================
# 🏆 IEEE 118-Bus R-PINN Robustness Test - Ultimate Platinum Edition
# 变量控制：【保留】完全继承主实验的结构、Mean-Loss类、300轮调度、动态物理权重与硬锚定
# 变量控制：【改动】仅在训练阶段的 PMU 观测数据上，精准注入 3% 高斯白噪声 (V乘性，Theta加性)
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandapower.networks as nw
import pandapower as pp
import matplotlib.pyplot as plt
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定 (严格对齐主模型)
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_NOISE_LEVEL = 0.01  # 测试 1% 极限噪声

print(f"🚀 [Robustness Test] 118节点抗噪实验就绪 | 算力: {device} | 注入噪声: {TEST_NOISE_LEVEL*100}%")

NUM_NODES = 118
obs_indices = [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]

# ------------------------------------------
# 2. 物理矩阵与节点身份掩码 (严格对齐主模型)
# ------------------------------------------
net = nw.case118()
pp.runpp(net)
Ybus = net._ppc['internal']['Ybus']
G_tensor = torch.tensor(Ybus.real.toarray(), dtype=torch.float32).to(device)
B_tensor = torch.tensor(Ybus.imag.toarray(), dtype=torch.float32).to(device)

slack_buses = net.ext_grid.bus.values      
pv_buses = net.gen.bus.values
load_buses = net.load.bus.values
all_buses = np.arange(NUM_NODES)
zero_inj_buses = np.setdiff1d(all_buses, np.concatenate([slack_buses, pv_buses, load_buses]))

mask_p_np = np.ones(NUM_NODES)
mask_p_np[slack_buses] = 0.0

mask_q_np = np.zeros(NUM_NODES)
mask_q_np[load_buses] = 1.0
mask_q_np[zero_inj_buses] = 1.0

mask_p = torch.tensor(mask_p_np, dtype=torch.float32).to(device)
mask_q = torch.tensor(mask_q_np, dtype=torch.float32).to(device)

# ------------------------------------------
# 3. 模型与损失封装 (严格对齐主模型)
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    # ✅ 修复：直接计算角度差，绝不能再乘 pi/180！
    dT = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=118):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        theta_pred = out[:, self.node_num:] * 0.05 + 0.5236    
        vm_pred = vm_pred.clone(); theta_pred = theta_pred.clone()
        vm_pred[:, 68] = 1.035; theta_pred[:, 68] = 0.5236      # 硬锚定
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        diff_p = (P_calc - P_real) ** 2
        diff_q = (Q_calc - Q_real) ** 2
        bs = diff_p.shape[0]
        P_loss = torch.sum(diff_p * mask_p) / (torch.sum(mask_p) * bs)
        Q_loss = torch.sum(diff_q * mask_q) / (torch.sum(mask_q) * bs)
        
        obs_loss_v = self.mse(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        obs_loss = obs_loss_v + obs_loss_t
        
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.10)
        penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
        
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_idx_list, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_idx_list:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载 (严格对齐主模型)
# ------------------------------------------
data_3d = np.load(r'ieee118_dataset/ieee118_data_50k.npy')
P_inj = -data_3d[:, :, 0] / 100.0  
Q_inj = -data_3d[:, :, 1] / 100.0
V_val = data_3d[:, :, 2]
T_rad = np.deg2rad(data_3d[:, :, 3]) 

X_raw = np.concatenate([P_inj, Q_inj], axis=1)
Y_label = np.concatenate([V_val, T_rad], axis=1)

train_size = 40000
X_tr_raw, X_te_raw = X_raw[:train_size], X_raw[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_inj[:train_size], Q_inj[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

class GridDataset(Dataset):
    def __init__(self, x, y, p, q): self.x, self.y, self.p, self.q = x, y, p, q
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.p[i], self.q[i]

train_loader = DataLoader(GridDataset(
    torch.tensor(X_tr_norm, dtype=torch.float32), torch.tensor(Y_tr, dtype=torch.float32),
    torch.tensor(P_tr_phys, dtype=torch.float32), torch.tensor(Q_tr_phys, dtype=torch.float32)), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 训练 (核心变量控制：仅在观测量注入噪声)
# ------------------------------------------
model = PowerGridPINN(node_num=118).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print(f"🔥 开始 300 轮训练 (强制投毒 {TEST_NOISE_LEVEL*100}% 高斯噪声)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        # ☠️ 投毒环节：克隆真实标签，精准在观测点注入噪声
        noisy_by = by.clone()
        
        # 1. 乘性噪声注入电压 (V)
        noise_v = torch.randn_like(noisy_by[:, obs_indices]) * TEST_NOISE_LEVEL
        noisy_by[:, obs_indices] = noisy_by[:, obs_indices] * (1 + noise_v)
        
        # 2. 加性噪声注入相角 (Theta)，偏移量为 118
        noise_t = torch.randn_like(noisy_by[:, 118 + np.array(obs_indices)]) * TEST_NOISE_LEVEL
        noisy_by[:, 118 + np.array(obs_indices)] = noisy_by[:, 118 + np.array(obs_indices)] + noise_t
        
        # 将被污染的 noisy_by 喂给损失函数，考验模型的物理纠错能力
        loss = loss_fn(vp, tp, bp, bq, noisy_by[:, :118], noisy_by[:, 118:], lambda_p, obs_weight=5e6)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 测试集干净数据评估
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.tensor(X_te_norm, dtype=torch.float32).to(device)
    ty = torch.tensor(Y_te, dtype=torch.float32).to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all = ty[:, :118]
    t_true_all = ty[:, 118:]
    
    err_v = (v_pred_all - v_true_all).cpu().numpy()
    err_t = (t_pred_all - t_true_all).cpu().numpy()
    mae_v = np.mean(np.abs(err_v))
    rmse_v = np.sqrt(np.mean(err_v**2))
    mae_t = np.mean(np.abs(err_t))
    rmse_t = np.sqrt(np.mean(err_t**2))

print("\n" + "="*65)
print(f"🏆 [Robustness Test - {TEST_NOISE_LEVEL*100}% Noise] 测试集对账单")
print(f"🌍 Voltage (V) MAE  : {mae_v:.6e} p.u. | RMSE: {rmse_v:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t:.6e} rad  | RMSE: {rmse_t:.6e} rad")
print("="*65)

# ------------------------------------------
# 7. 绘图输出
# ------------------------------------------
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-paper')

sid = 10
v_t_plot = v_true_all[sid].cpu().numpy()
v_p_plot = v_pred_all[sid].cpu().numpy()
t_t_plot = t_true_all[sid].cpu().numpy()
t_p_plot = t_pred_all[sid].cpu().numpy()
nodes = np.arange(118)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), dpi=600, sharex=True)
ax1.axvspan(0, 117, color='gray', alpha=0.05, label='Blind Zone (85%)')
ax1.plot(nodes, v_t_plot, 'k-', label='Ground Truth (Clean)', linewidth=2.0)
ax1.plot(nodes, v_p_plot, 'r--', label=f'Model w/ {TEST_NOISE_LEVEL*100}% Noise', linewidth=1.5)
ax1.scatter(obs_indices, v_t_plot[obs_indices], color='blue', marker='*', s=150, label='PMU Sensors')
ax1.set_title(f"Robustness Test ({TEST_NOISE_LEVEL*100}% Noise) - Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', fontsize=11)

ax2.axvspan(0, 117, color='gray', alpha=0.05)
ax2.plot(nodes, t_t_plot, 'k-', linewidth=2.0)
ax2.plot(nodes, t_p_plot, 'b--', label=f'Model w/ {TEST_NOISE_LEVEL*100}% Noise Theta', linewidth=1.5)
ax2.scatter(obs_indices, t_t_plot[obs_indices], color='blue', marker='*', s=150)
ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} rad | RMSE: {rmse_t:.4e} rad)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Bus Index", fontsize=12)
ax2.set_ylabel("Phase Angle (rad)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('ieee118_robustness_3pct_noise.png', bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ==============================================================================
# 🏆 IEEE 118-Bus R-PINN Robustness Test - Ultimate Platinum Edition
# 变量控制：【保留】完全继承主实验的结构、Mean-Loss类、300轮调度、动态物理权重与硬锚定
# 变量控制：【改动】仅在训练阶段的 PMU 观测数据上，精准注入 3% 高斯白噪声 (V乘性，Theta加性)
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandapower.networks as nw
import pandapower as pp
import matplotlib.pyplot as plt
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定 (严格对齐主模型)
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_NOISE_LEVEL = 0.02  # 测试 2% 极限噪声

print(f"🚀 [Robustness Test] 118节点抗噪实验就绪 | 算力: {device} | 注入噪声: {TEST_NOISE_LEVEL*100}%")

NUM_NODES = 118
obs_indices = [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]

# ------------------------------------------
# 2. 物理矩阵与节点身份掩码 (严格对齐主模型)
# ------------------------------------------
net = nw.case118()
pp.runpp(net)
Ybus = net._ppc['internal']['Ybus']
G_tensor = torch.tensor(Ybus.real.toarray(), dtype=torch.float32).to(device)
B_tensor = torch.tensor(Ybus.imag.toarray(), dtype=torch.float32).to(device)

slack_buses = net.ext_grid.bus.values      
pv_buses = net.gen.bus.values
load_buses = net.load.bus.values
all_buses = np.arange(NUM_NODES)
zero_inj_buses = np.setdiff1d(all_buses, np.concatenate([slack_buses, pv_buses, load_buses]))

mask_p_np = np.ones(NUM_NODES)
mask_p_np[slack_buses] = 0.0

mask_q_np = np.zeros(NUM_NODES)
mask_q_np[load_buses] = 1.0
mask_q_np[zero_inj_buses] = 1.0

mask_p = torch.tensor(mask_p_np, dtype=torch.float32).to(device)
mask_q = torch.tensor(mask_q_np, dtype=torch.float32).to(device)

# ------------------------------------------
# 3. 模型与损失封装 (严格对齐主模型)
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    # ✅ 修复：直接计算角度差，绝不能再乘 pi/180！
    dT = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=118):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        theta_pred = out[:, self.node_num:] * 0.05 + 0.5236    
        vm_pred = vm_pred.clone(); theta_pred = theta_pred.clone()
        vm_pred[:, 68] = 1.035; theta_pred[:, 68] = 0.5236      # 硬锚定
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        diff_p = (P_calc - P_real) ** 2
        diff_q = (Q_calc - Q_real) ** 2
        bs = diff_p.shape[0]
        P_loss = torch.sum(diff_p * mask_p) / (torch.sum(mask_p) * bs)
        Q_loss = torch.sum(diff_q * mask_q) / (torch.sum(mask_q) * bs)
        
        obs_loss_v = self.mse(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        obs_loss = obs_loss_v + obs_loss_t
        
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.10)
        penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
        
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_idx_list, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_idx_list:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载 (严格对齐主模型)
# ------------------------------------------
data_3d = np.load(r'ieee118_dataset/ieee118_data_50k.npy')
P_inj = -data_3d[:, :, 0] / 100.0  
Q_inj = -data_3d[:, :, 1] / 100.0
V_val = data_3d[:, :, 2]
T_rad = np.deg2rad(data_3d[:, :, 3]) 

X_raw = np.concatenate([P_inj, Q_inj], axis=1)
Y_label = np.concatenate([V_val, T_rad], axis=1)

train_size = 40000
X_tr_raw, X_te_raw = X_raw[:train_size], X_raw[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_inj[:train_size], Q_inj[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

class GridDataset(Dataset):
    def __init__(self, x, y, p, q): self.x, self.y, self.p, self.q = x, y, p, q
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.p[i], self.q[i]

train_loader = DataLoader(GridDataset(
    torch.tensor(X_tr_norm, dtype=torch.float32), torch.tensor(Y_tr, dtype=torch.float32),
    torch.tensor(P_tr_phys, dtype=torch.float32), torch.tensor(Q_tr_phys, dtype=torch.float32)), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 训练 (核心变量控制：仅在观测量注入噪声)
# ------------------------------------------
model = PowerGridPINN(node_num=118).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print(f"🔥 开始 300 轮训练 (强制投毒 {TEST_NOISE_LEVEL*100}% 高斯噪声)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        # ☠️ 投毒环节：克隆真实标签，精准在观测点注入噪声
        noisy_by = by.clone()
        
        # 1. 乘性噪声注入电压 (V)
        noise_v = torch.randn_like(noisy_by[:, obs_indices]) * TEST_NOISE_LEVEL
        noisy_by[:, obs_indices] = noisy_by[:, obs_indices] * (1 + noise_v)
        
        # 2. 加性噪声注入相角 (Theta)，偏移量为 118
        noise_t = torch.randn_like(noisy_by[:, 118 + np.array(obs_indices)]) * TEST_NOISE_LEVEL
        noisy_by[:, 118 + np.array(obs_indices)] = noisy_by[:, 118 + np.array(obs_indices)] + noise_t
        
        # 将被污染的 noisy_by 喂给损失函数，考验模型的物理纠错能力
        loss = loss_fn(vp, tp, bp, bq, noisy_by[:, :118], noisy_by[:, 118:], lambda_p, obs_weight=5e6)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 测试集干净数据评估
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.tensor(X_te_norm, dtype=torch.float32).to(device)
    ty = torch.tensor(Y_te, dtype=torch.float32).to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all = ty[:, :118]
    t_true_all = ty[:, 118:]
    
    err_v = (v_pred_all - v_true_all).cpu().numpy()
    err_t = (t_pred_all - t_true_all).cpu().numpy()
    mae_v = np.mean(np.abs(err_v))
    rmse_v = np.sqrt(np.mean(err_v**2))
    mae_t = np.mean(np.abs(err_t))
    rmse_t = np.sqrt(np.mean(err_t**2))

print("\n" + "="*65)
print(f"🏆 [Robustness Test - {TEST_NOISE_LEVEL*100}% Noise] 测试集对账单")
print(f"🌍 Voltage (V) MAE  : {mae_v:.6e} p.u. | RMSE: {rmse_v:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t:.6e} rad  | RMSE: {rmse_t:.6e} rad")
print("="*65)

# ------------------------------------------
# 7. 绘图输出
# ------------------------------------------
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-paper')

sid = 10
v_t_plot = v_true_all[sid].cpu().numpy()
v_p_plot = v_pred_all[sid].cpu().numpy()
t_t_plot = t_true_all[sid].cpu().numpy()
t_p_plot = t_pred_all[sid].cpu().numpy()
nodes = np.arange(118)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), dpi=600, sharex=True)
ax1.axvspan(0, 117, color='gray', alpha=0.05, label='Blind Zone (85%)')
ax1.plot(nodes, v_t_plot, 'k-', label='Ground Truth (Clean)', linewidth=2.0)
ax1.plot(nodes, v_p_plot, 'r--', label=f'Model w/ {TEST_NOISE_LEVEL*100}% Noise', linewidth=1.5)
ax1.scatter(obs_indices, v_t_plot[obs_indices], color='blue', marker='*', s=150, label='PMU Sensors')
ax1.set_title(f"Robustness Test ({TEST_NOISE_LEVEL*100}% Noise) - Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', fontsize=11)

ax2.axvspan(0, 117, color='gray', alpha=0.05)
ax2.plot(nodes, t_t_plot, 'k-', linewidth=2.0)
ax2.plot(nodes, t_p_plot, 'b--', label=f'Model w/ {TEST_NOISE_LEVEL*100}% Noise Theta', linewidth=1.5)
ax2.scatter(obs_indices, t_t_plot[obs_indices], color='blue', marker='*', s=150)
ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} rad | RMSE: {rmse_t:.4e} rad)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Bus Index", fontsize=12)
ax2.set_ylabel("Phase Angle (rad)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('ieee118_robustness_3pct_noise.png', bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ==============================================================================
# 🏆 IEEE 118-Bus R-PINN Robustness Test - Ultimate Platinum Edition
# 变量控制：【保留】完全继承主实验的结构、Mean-Loss类、300轮调度、动态物理权重与硬锚定
# 变量控制：【改动】仅在训练阶段的 PMU 观测数据上，精准注入 3% 高斯白噪声 (V乘性，Theta加性)
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandapower.networks as nw
import pandapower as pp
import matplotlib.pyplot as plt
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定 (严格对齐主模型)
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_NOISE_LEVEL = 0.03  # 测试 3% 极限噪声

print(f"🚀 [Robustness Test] 118节点抗噪实验就绪 | 算力: {device} | 注入噪声: {TEST_NOISE_LEVEL*100}%")

NUM_NODES = 118
obs_indices = [0, 6, 13, 20, 27, 34, 41, 48, 55, 61, 68, 75, 82, 89, 96, 103, 110, 117]

# ------------------------------------------
# 2. 物理矩阵与节点身份掩码 (严格对齐主模型)
# ------------------------------------------
net = nw.case118()
pp.runpp(net)
Ybus = net._ppc['internal']['Ybus']
G_tensor = torch.tensor(Ybus.real.toarray(), dtype=torch.float32).to(device)
B_tensor = torch.tensor(Ybus.imag.toarray(), dtype=torch.float32).to(device)

slack_buses = net.ext_grid.bus.values      
pv_buses = net.gen.bus.values
load_buses = net.load.bus.values
all_buses = np.arange(NUM_NODES)
zero_inj_buses = np.setdiff1d(all_buses, np.concatenate([slack_buses, pv_buses, load_buses]))

mask_p_np = np.ones(NUM_NODES)
mask_p_np[slack_buses] = 0.0

mask_q_np = np.zeros(NUM_NODES)
mask_q_np[load_buses] = 1.0
mask_q_np[zero_inj_buses] = 1.0

mask_p = torch.tensor(mask_p_np, dtype=torch.float32).to(device)
mask_q = torch.tensor(mask_q_np, dtype=torch.float32).to(device)

# ------------------------------------------
# 3. 模型与损失封装 (严格对齐主模型)
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    # ✅ 修复：直接计算角度差，绝不能再乘 pi/180！
    dT = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=118):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        theta_pred = out[:, self.node_num:] * 0.05 + 0.5236    
        vm_pred = vm_pred.clone(); theta_pred = theta_pred.clone()
        vm_pred[:, 68] = 1.035; theta_pred[:, 68] = 0.5236      # 硬锚定
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        diff_p = (P_calc - P_real) ** 2
        diff_q = (Q_calc - Q_real) ** 2
        bs = diff_p.shape[0]
        P_loss = torch.sum(diff_p * mask_p) / (torch.sum(mask_p) * bs)
        Q_loss = torch.sum(diff_q * mask_q) / (torch.sum(mask_q) * bs)
        
        obs_loss_v = self.mse(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        obs_loss = obs_loss_v + obs_loss_t
        
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.10)
        penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
        
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_idx_list, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_idx_list:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载 (严格对齐主模型)
# ------------------------------------------
data_3d = np.load(r'ieee118_dataset/ieee118_data_50k.npy')
P_inj = -data_3d[:, :, 0] / 100.0  
Q_inj = -data_3d[:, :, 1] / 100.0
V_val = data_3d[:, :, 2]
T_rad = np.deg2rad(data_3d[:, :, 3]) 

X_raw = np.concatenate([P_inj, Q_inj], axis=1)
Y_label = np.concatenate([V_val, T_rad], axis=1)

train_size = 40000
X_tr_raw, X_te_raw = X_raw[:train_size], X_raw[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_inj[:train_size], Q_inj[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

class GridDataset(Dataset):
    def __init__(self, x, y, p, q): self.x, self.y, self.p, self.q = x, y, p, q
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.p[i], self.q[i]

train_loader = DataLoader(GridDataset(
    torch.tensor(X_tr_norm, dtype=torch.float32), torch.tensor(Y_tr, dtype=torch.float32),
    torch.tensor(P_tr_phys, dtype=torch.float32), torch.tensor(Q_tr_phys, dtype=torch.float32)), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 训练 (核心变量控制：仅在观测量注入噪声)
# ------------------------------------------
model = PowerGridPINN(node_num=118).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print(f"🔥 开始 300 轮训练 (强制投毒 {TEST_NOISE_LEVEL*100}% 高斯噪声)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        # ☠️ 投毒环节：克隆真实标签，精准在观测点注入噪声
        noisy_by = by.clone()
        
        # 1. 乘性噪声注入电压 (V)
        noise_v = torch.randn_like(noisy_by[:, obs_indices]) * TEST_NOISE_LEVEL
        noisy_by[:, obs_indices] = noisy_by[:, obs_indices] * (1 + noise_v)
        
        # 2. 加性噪声注入相角 (Theta)，偏移量为 118
        noise_t = torch.randn_like(noisy_by[:, 118 + np.array(obs_indices)]) * TEST_NOISE_LEVEL
        noisy_by[:, 118 + np.array(obs_indices)] = noisy_by[:, 118 + np.array(obs_indices)] + noise_t
        
        # 将被污染的 noisy_by 喂给损失函数，考验模型的物理纠错能力
        loss = loss_fn(vp, tp, bp, bq, noisy_by[:, :118], noisy_by[:, 118:], lambda_p, obs_weight=5e6)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 测试集干净数据评估
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.tensor(X_te_norm, dtype=torch.float32).to(device)
    ty = torch.tensor(Y_te, dtype=torch.float32).to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all = ty[:, :118]
    t_true_all = ty[:, 118:]
    
    err_v = (v_pred_all - v_true_all).cpu().numpy()
    err_t = (t_pred_all - t_true_all).cpu().numpy()
    mae_v = np.mean(np.abs(err_v))
    rmse_v = np.sqrt(np.mean(err_v**2))
    mae_t = np.mean(np.abs(err_t))
    rmse_t = np.sqrt(np.mean(err_t**2))

print("\n" + "="*65)
print(f"🏆 [Robustness Test - {TEST_NOISE_LEVEL*100}% Noise] 测试集对账单")
print(f"🌍 Voltage (V) MAE  : {mae_v:.6e} p.u. | RMSE: {rmse_v:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t:.6e} rad  | RMSE: {rmse_t:.6e} rad")
print("="*65)

# ------------------------------------------
# 7. 绘图输出
# ------------------------------------------
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-paper')

sid = 10
v_t_plot = v_true_all[sid].cpu().numpy()
v_p_plot = v_pred_all[sid].cpu().numpy()
t_t_plot = t_true_all[sid].cpu().numpy()
t_p_plot = t_pred_all[sid].cpu().numpy()
nodes = np.arange(118)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), dpi=600, sharex=True)
ax1.axvspan(0, 117, color='gray', alpha=0.05, label='Blind Zone (85%)')
ax1.plot(nodes, v_t_plot, 'k-', label='Ground Truth (Clean)', linewidth=2.0)
ax1.plot(nodes, v_p_plot, 'r--', label=f'Model w/ {TEST_NOISE_LEVEL*100}% Noise', linewidth=1.5)
ax1.scatter(obs_indices, v_t_plot[obs_indices], color='blue', marker='*', s=150, label='PMU Sensors')
ax1.set_title(f"Robustness Test ({TEST_NOISE_LEVEL*100}% Noise) - Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', fontsize=11)

ax2.axvspan(0, 117, color='gray', alpha=0.05)
ax2.plot(nodes, t_t_plot, 'k-', linewidth=2.0)
ax2.plot(nodes, t_p_plot, 'b--', label=f'Model w/ {TEST_NOISE_LEVEL*100}% Noise Theta', linewidth=1.5)
ax2.scatter(obs_indices, t_t_plot[obs_indices], color='blue', marker='*', s=150)
ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} rad | RMSE: {rmse_t:.4e} rad)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Bus Index", fontsize=12)
ax2.set_ylabel("Phase Angle (rad)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('ieee118_robustness_3pct_noise.png', bbox_inches='tight', dpi=600)
plt.show()